**Author:** Amanda Baright

**Date:** 04.28.2026

**Purpose:** ST 554 Final Project

# ST 554 Final Project

This final project will assess my ability to use spark to handle streaming data, use spark for fitting a machine learning model, and a few other things. The project will be split into three main components: Fitting a machine learning model using `pyspark`'s `MLlib` module, reading in a stream of data that is produced ourselves using a `.py` file, and using the model to do predictions on the stream and write the predictions out to the console.

The data is modified from the [UCI machine learning repository](https://archive.ics.uci.edu/dataset/849/power+consumption+of+tetouan+city), where the study was about relating power consumption from different time zones of Tetouan city to various factors like time of day, temperatrue, and humidity. We will treat the `Power_Zone_3` variable as our response variable, and will use all of the other varialbes as predictors. This will allow us to predict the value of `Power_Zone_3` appropriately if that reading goes offline in the future.

We are provided a chunk of the modified data to build our model on, and will then "stream data" to a folder in the form of CSV files. We will monitor this folder, and will use the fitted model to predict on the incoming data.

## Fitting Your Model

This first part will focus on model fitting. We will use the `power_ml_data.csv` which is available at: https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv. We first want to read in this data using a standard pandas data frame using `pd.read_csv()`. We will also use this first step to read in all of the libraries we may need. We will also initialize a Spark session and convert this pandas data frame into a spark data frame.

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import (SQLTransformer, Binarizer, OneHotEncoder, 
                                VectorAssembler, PCA, StandardScaler)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

In [3]:
spark = SparkSession.builder.getOrCreate()

# Read data into pandas
pandas_df = pd.read_csv("power_ml_data.csv")

# Convert to spark data frame
power_df = spark.createDataFrame(pandas_df)

# Look at first few rows
power_df.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

We then want to start building a `MLlib` Pipeline that includes all of the pre-specified transformations.

The first step will be to use a SQL transformer to rename `Power_Zone_3` as `label` and cast the `Hour` column to be stored as a `DoubleType` variable.

In [4]:
sqlTrans = SQLTransformer(
    statement="SELECT *, Power_Zone_3 AS label, CAST(Hour AS DOUBLE) AS HourDouble FROM __THIS__"
)

We then want to add a component that binarizes the `Hour` column based on the column being less than 6.5 or not, which essentially breaks the `Hour` column into night vs day.

In [5]:
binarizer = Binarizer(threshold=6.5, inputCol="HourDouble", outputCol="binHour")

We then want to do one-hot encoding for the `Month` column, where each unique month in the `Month` column is transformed into its own binary column with 1 indicating the presence of that month category and 0 indicates the absence.

In [6]:
encoder = OneHotEncoder(inputCols=["Month"], outputCols=["Month_vec"])

We now want to use `VectorAssembler()` to assemble the weather related variables (`Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows`). Additionally, we will want to standardize the weather variables before doing the PCA fit, as PCA is sensitive to the scale of the data.

In [7]:
weather_assembler = VectorAssembler(
    inputCols=["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"],
    outputCol="weather_features"
)

scaler_pca = StandardScaler(inputCol="weather_features", outputCol="scaled_weather", 
                            withStd=True, withMean=True)

Now that we standardized our weather features, we can run a PCA fit using two PCs in the transformation.

In [8]:
pca = PCA(k=2, inputCol="scaled_weather", outputCol="pca_features")

We then do a final feature assembly where we include the `pca_features` that were selected with the PCA fit, the binary `Hour` variable, `Power_Zone_1`, `Power_Zone_2`, and the `Month` indicator variable. These will be stored into a `unscaled_features` column by using the `VectorAssembler()` method again. We then want to standardize the final features for the Elastic Net Model, as the Elastic Net model adds a penalty term to the magnitude of the coefficients so features with larger scales will have unfairly small coefficients which may lead to some predictors to be ignored.

In [9]:
final_assembler = VectorAssembler(
    inputCols=["pca_features", "binHour", "Power_Zone_1", "Power_Zone_2", "Month_vec"],
    outputCol="unscaled_features"
)

scaler_final = StandardScaler(inputCol="unscaled_features", outputCol="features", 
                              withStd=True, withMean=True)

Now we can finally define the pipeline with all of these amazing transformations!

In [10]:
pipeline = Pipeline(stages=[
    sqlTrans, 
    binarizer, 
    encoder, 
    weather_assembler, 
    scaler_pca, 
    pca, 
    final_assembler, 
    scaler_final
])

We then want to use the `CrossValidator()` function with the `LinearRegression()` function to fit an elastic net model. We will use a pretty extensive grid with the `ParamGridBuilder()` function to examine all combinations of a few parameters for `regParam` and `elasticNetParam`. We will also fit the model using 5-fold CR with Root Mean Square Error (`rmse`) as our criterion.

In [11]:
# Set up the LinearRegression() function
lr = LinearRegression(featuresCol="features", labelCol="label")

# Define the extensive grid selection
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

# Set up 5-fold CV with rmse as our metric of choice.
cv = CrossValidator(estimator=lr,
                    estimatorParamMaps=paramGrid,
                    evaluator=RegressionEvaluator(metricName="rmse"),
                    numFolds=5)

Now we can finally fit the entire workflow with all of our components built above. The first step is to fit the pipeline to the data.

In [12]:
pipelineModel = pipeline.fit(power_df)

Next we will transform our data using our pipeline to get the `features` and `label` columns.

In [13]:
transformedData = pipelineModel.transform(power_df)

We can now fit the `CrossValidator()` on the transformed data.

In [14]:
cvModel = cv.fit(transformedData)

This code runs without visible warnings.


26/04/29 14:23:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/29 14:23:37 WARN Instrumentation: [256261b9] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 14:23:39 WARN Instrumentation: [256261b9] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/29 14:23:40 WARN Instrumentation: [4438bbc6] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 14:23:40 WARN Instrumentation: [4438bbc6] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/29 14:23:41 WARN Instrumentation: [9ff94e7a] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 14:23:41 WARN Instrumentation: [9ff94e7a] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

We then want to look at the best model and report the optimal values chosen for the tuning parameters and the CV error. Here we can extract the best model with the `.bestModel` attribute. We can then print out the `regParam` and `elasticNetParam` with their respective `.getParam()` method and find the average RMSE from the Cross Validation run to get the CV error.

In [15]:
bestModel = cvModel.bestModel
print(f"Optimal regParam: {bestModel.getRegParam()}")
print(f"Optimal elasticNetParam: {bestModel.getElasticNetParam()}")
print(f"CV RMSE: {min(cvModel.avgMetrics)}")

Optimal regParam: 0.05
Optimal elasticNetParam: 0.1
CV RMSE: 2175.1398303619526


We now want to report the training RMSE using a similar method as done in the ST 554 course notes. This includes using the fitted model as a transformer and evaluating on the entire training set. Here we will find the predictions using the `.transform()` method on the transformed data. We can then use the `RegressionEvaluator()` function to evaluate the fit using RMSE as the metric.

In [16]:
predictions = cvModel.transform(transformedData)
evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
print(f"Training RMSE: {evaluator.evaluate(predictions)}")

Training RMSE: 2174.448931496749


We then want to take the outputted transformations from the model (i.e. the predictions) and create a `residual` column. A residual is the observation (`label`) minus the prediction (`prediction`). We can utilize the `.withColumn()` method to help create this new `residual` column. We then will print out the data frame with these `residual`s, the `label` column, and the `predictions`. Here we can use `.select()` with `.show()` to print out these columns.

In [17]:
predictions = predictions.withColumn("residual", predictions.label - predictions.prediction)
predictions.select("label", "prediction", "residual").show()

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20935.222626646493|-694.2587666464933|
|20131.08434| 18701.55495877999|1429.5293812200107|
|19668.43373| 18237.19183322843|1431.2418967715712|
|18899.27711| 17615.91729995449|1283.3598100455092|
|18442.40964|17017.450559678364|1424.9590803216379|
|18130.12048|16540.710194816464|1589.4102851835378|
|17945.06024|16114.799730385681| 1830.260509614318|
|17459.27711|15742.177027126962|1717.1000828730375|
|17025.54217|15282.720684270233| 1742.821485729768|
|16794.21687|14935.081895139816|1859.1349748601842|
|16638.07229|14645.418124495305| 1992.654165504695|
|16395.18072|14395.851802230101| 1999.328917769899|
|16117.59036|14073.986054534807|2043.6043054651927|
| 15822.6506| 13609.49733950086|2213.1532604991407|
|15672.28916|13432.419802348406|2239.8693576515943|
|15597.10843|13282.510312025239|2314.5981179747614|
|15510.36145

## Streaming Part

We now want to start the streaming part of the final project. Here we have another file `power_streaming_data.csv` that is available at: https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv. We will want to ensure it's stored where our `.py` file we create later will find it. We will randomly sample rows from this file to output to `.csv` files that we will be reading in.

### Reading a Stream

We're going to read in a stream in the form of `.csv` files. We will want a folder where we will be sending these created `.csv` files. Here I created a folder called `streaming_folder` for convenience sake. However, our first step for this step will be setting up the schema for the stream, where we will just use the schema from the original data for simplicity sake.

In [34]:
from pyspark.sql.types import StructType, DoubleType, IntegerType
import pyspark.sql.functions as F

schema = StructType() \
    .add("Temperature", DoubleType()) \
    .add("Humidity", DoubleType()) \
    .add("Wind_Speed", DoubleType()) \
    .add("General_Diffuse_Flows", DoubleType()) \
    .add("Diffuse_Flows", DoubleType()) \
    .add("Power_Zone_1", DoubleType()) \
    .add("Power_Zone_2", DoubleType()) \
    .add("Power_Zone_3", DoubleType()) \
    .add("Month", IntegerType()) \
    .add("Hour", IntegerType())

We then want to set up the `readStream`, making sure we include the `header = True` as we will output files with a header and don't need to read that in.

In [35]:
streaming_df = spark.readStream \
    .schema(schema) \
    .option("header", "true") \
    .csv("streaming_folder")

### Transform / Aggregation Step

Now we will do two separate things on the stream and join them together!

With the stream, we will use the model transformer to obtain predictions from the incoming data. On the resulting predicitons we will also create a `residual` column as done in the previous section where we will only return the `label`, `prediction`, and `residual` column.

In [36]:
# We use the fitted models from the "Fitting Your Model" section to get the predictions
pipeline_transformed = pipelineModel.transform(streaming_df)
predictions_df = cvModel.transform(pipeline_transformed)

stream_1 = predictions_df.withColumn("residual", F.col("label") - F.col("prediction")) \
    .select("label", "prediction", "residual")

We then want to use another transformation on the (original) stream, to modify the response variable to be called `label` to help facilitate the join. That is, we will use the `streaming_df` and modify `Power_Zone_3` to be called `label` by using `withColumnRenamed()`.

In [37]:
stream_2 = streaming_df.withColumnRenamed("Power_Zone_3", "label") \
    .select("label", "Temperature", "Humidity", "Wind_Speed", 
            "General_Diffuse_Flows", "Diffuse_Flows", "Power_Zone_1", 
            "Power_Zone_2", "Month", "Hour")

We then want to join the above transformations with an inner join based on the `label` variable, as it is common to both streams.

In [43]:
final_stream = stream_1.join(stream_2, "label")

### Writing Step

We now want to write the stream to the console using the `append` output mode and start the query!

In [44]:
query = final_stream.writeStream \
    .outputMode("append") \
    .format("console") \
    .trigger(processingTime='10 seconds') \
    .start()

26/04/29 15:04:06 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-88330a76-9517-431d-b32f-6f39621e4e46. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/29 15:04:06 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/29 15:05:06 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000} milliseconds, but spent 46251 milliseconds


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|10907.23404|10693.777383986278| 213.45665601372093|      17.67|    77.2|     0.081|                0.084|        0.126| 26972.42888| 17622.82158|   10|   6|
|13084.01216|14358.645554486364|-1274.6333944863636|      23.86|   61.86|      0.07|                209.4|        179.0| 36620.74398| 23676.34855|   10|  14|
|10135.89436| 11468.60461850867|-1332.7102585086704|       17.1|   37.53|     0.091|                411.9|       

26/04/29 15:05:28 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000} milliseconds, but spent 22135 milliseconds


-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15735.06073|16290.611216504296|-555.5504865042967|      16.96|    76.7|     4.923|                0.051|        0.148| 26716.32787| 17030.34056|    5|   1|
|13162.40964|12695.568384168417| 466.8412558315831|      12.28|   64.34|      0.08|                 83.4|         81.0| 25342.78481| 12164.13374|    1|   9|
| 16840.0402|17133.859451252454|-293.8192512524547|      13.63|   67.12|     0.079|                126.6|        127.3

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|9657.623049| 6754.250362011191| 2903.3726869888087|      11.66|    53.3|     0.078|                0.095|        0.082| 20720.91255| 17589.44461|   12|   3|
|17912.12308|19516.418398056307|-1604.2953180563054|      23.65|   55.48|     0.071|                720.0|        58.94|  35081.3245| 21113.51351|    6|  15|
|10709.63563| 13565.01491907975|-2855.3792890797486|      16.53|    89.8|     4.917|                41.02|       

...

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|12802.59109|12438.916378822996| 363.6747111770037|      18.63|    86.5|     0.065|                0.073|        0.074| 22070.55738| 12208.04954|    5|   5|
|    12000.0|11174.276544912027|  825.723455087973|      15.68|   56.17|     0.083|                0.033|        0.085| 27292.30769| 22310.33058|   11|  23|
| 10654.5018| 9298.524990867018|1355.9768091329825|      16.22|   59.56|     0.076|                296.4|        202.

26/04/29 15:13:31 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000} milliseconds, but spent 11028 milliseconds


-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|37382.82132|32658.820730572566| 4724.000589427436|       30.8|   21.47|     4.907|                0.143|        0.126| 48592.40844|  33559.4509|    8|  20|
|14428.91566|13461.181453427213|  967.734206572788|      10.69|   59.09|     0.085|                0.073|          0.1| 22517.46835| 13670.51672|    1|   3|
|27193.10769|26578.426712064032| 614.6809779359683|      21.59|   59.88|     0.082|                19.94|        16.6

26/04/29 15:14:01 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000} milliseconds, but spent 11411 milliseconds


-------------------------------------------
Batch: 19
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|19874.33198| 18714.89089867442| 1159.4410813255781|      27.51|    41.5|      0.08|                437.4|        115.7| 36020.45902| 20931.26935|    5|  17|
|20165.82996|21122.951487087725| -957.1215270877256|      21.08|    77.1|     4.917|                141.1|        151.1| 37707.54098| 23442.72446|    5|  19|
|17395.66265|13309.540470008109| 4086.1221799918894|      22.12|    53.1|     0.075|                294.0|      

In [45]:
query.stop()

26/04/29 15:14:40 WARN DAGScheduler: Failed to cancel job group 31bcdb10-79db-404c-80e6-e2cc27380281. Cannot find active jobs for it.
26/04/29 15:14:40 WARN DAGScheduler: Failed to cancel job group 31bcdb10-79db-404c-80e6-e2cc27380281. Cannot find active jobs for it.


## Produce Data

We now will create a `.py` file called `produce_data.py` that will read the `power_streaming_data.csv` into a regular pandas data frame and does a few different things.

This `.py` file will create a loop that will have roughly 20 iterations. In the loop we will randomly sample five rows and output those to a `.csv` file in the `streaming_folder` we created earlier. We will pay special attention to not write out the indices. We will pause for 10 seconds in between outputting the data sets. This loop will be submitted in a python console. We should see that the output begins to appear in the notebook.